Esse notebook tem como objetivo aplicar uma função em cada partição no Spark com o MapPartitions. Essa função possibilita aplicar a mesma função, de forma paralela, em todas as partições, neste caso a partição contem os arquivos de audio que precisam ser processados.

In [6]:
from glob import glob
from time import sleep
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as pf

In [7]:
spark = (
    SparkSession.builder
    #.config("spark.executor.instances", "1")
    #.config("spark.executor.cores", "2")
    .getOrCreate()
)

wav_files = glob("../audio/*.wav")

spark

Eu tenho um computador com 12 cores, então vai ser mapeado 12 partições no DataFrame com o Spark, vou definir o paralelismo como 4 para testar usando o método coalesce(4).
No Databricks é possível usar o formato binaryFile para ler os arquivos, ou o CloudFiles para Spark Streaming.

In [8]:
df = spark.createDataFrame([(f,) for f in wav_files], ["file_path"])
df = df.coalesce(4)

Vou aplicar a função de transcrição em cada uma das partições, neste caso, de forma paralela nas 4 partições ao mesmo tempo.

In [9]:
def transcribe_audio(file_path):
    sleep(5)
    return "TEXT"

def process_partition(iterator):
    for row in iterator:
        file_path = row.file_path
        try:
            transcription = transcribe_audio(file_path)
            dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            yield (file_path, transcription, dt)
        except Exception as e:
            dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            yield (file_path, f"ERRO: {str(e)}", dt)

In [10]:
df = spark.createDataFrame(
    df.rdd.mapPartitions(process_partition),
    ["file_path", "transcription", "datetime"]
)

In [11]:
df.orderBy("datetime").show()

+--------------------+-------------+-------------------+
|           file_path|transcription|           datetime|
+--------------------+-------------+-------------------+
|../audio/audio13.wav|         TEXT|2026-03-18 09:37:04|
|../audio/audio11.wav|         TEXT|2026-03-18 09:37:04|
|../audio/audio12.wav|         TEXT|2026-03-18 09:37:04|
|../audio/audio23.wav|         TEXT|2026-03-18 09:37:04|
|../audio/audio21.wav|         TEXT|2026-03-18 09:37:09|
|../audio/audio22.wav|         TEXT|2026-03-18 09:37:09|
+--------------------+-------------+-------------------+

